# Inference review: проверка адектватности инференса.

В этом ноутбуке предлопалается просмотр того, что модель генерирует что-то нормальное + заметр pass@1 на тесте.

## Пути и параметры

Надо указать три папки в первой ячейке:

- `PROJECT_PATH`: репозиторий с `lib/inference.py`.
- `EXPERIMENT_PATH`: полный путь к папке выбранного эксперимента; его надо задать явно. Это папка с `config.yaml`, `init_state/tokenizer.json` и `checkpoints`.
- `DATA_PATH`: папка `tokenized` с подкаталогом `val`.

`CHECKPOINT` выбирает `best.pt` или `last.pt`. Конфиг модели, токенайзер и веса читаются из одной папки эксперимента. Пути к обучающим данным из сохранённого конфига для загрузки модели не используются. Подходит папка из Kaggle Input или `/kaggle/working/experiment_runs/<имя>` после обучения.

In [ ]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
EXPERIMENT_PATH = ...  # TODO: полный путь к папке нужного эксперимента
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized").expanduser().resolve()
CHECKPOINT = "best"  # "best" или "last"


if EXPERIMENT_PATH is ...:
    raise ValueError("Надо задать EXPERIMENT_PATH: полный путь к папке эксперимента.")
EXPERIMENT_PATH = Path(EXPERIMENT_PATH).expanduser().resolve()

CONFIG_PATH = EXPERIMENT_PATH / "config.yaml"
TOKENIZER_PATH = EXPERIMENT_PATH / "init_state" / "tokenizer.json"
CHECKPOINT_PATH = EXPERIMENT_PATH / "checkpoints" / f"{CHECKPOINT}.pt"

VAL_PATH = DATA_PATH / "val"

In [ ]:
if CHECKPOINT not in ("best", "last"):
    raise ValueError('CHECKPOINT должен быть "best" или "last".')

for name, path in (("PROJECT_PATH", PROJECT_PATH), ("EXPERIMENT_PATH", EXPERIMENT_PATH)):
    if not path.is_dir():
        raise FileNotFoundError(f"{name}: папка не найдена: {path}")

required_files = [
    PROJECT_PATH / "lib" / "inference.py",
    CONFIG_PATH,
    TOKENIZER_PATH,
    CHECKPOINT_PATH,
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Некорректные пути в первой ячейке. Не найдены файлы:\n" + "\n".join(missing_files))

if not VAL_PATH.is_dir():
    raise FileNotFoundError(f"Папка валидации не найдена: {VAL_PATH}")

In [ ]:
print("Репозиторий:", PROJECT_PATH)
print("Эксперимент:", EXPERIMENT_PATH)
print("Конфиг:", CONFIG_PATH)
print("Токенайзер:", TOKENIZER_PATH)
print("Чекпойнт:", CHECKPOINT_PATH)
print("Валидация:", VAL_PATH)

In [ ]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb pygments

import os
import sys
from html import escape
from random import Random

os.environ["PROJECT_DIR"] = str(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))
for name in list(sys.modules):
    if name == "lib" or name.startswith("lib."):
        del sys.modules[name]

## Импорты

In [ ]:
from datasets import load_from_disk
from IPython.display import HTML, display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer

from lib.inference import generate_code, load_for_inference

### Красивый вывод

Он украден из ноутбука EDA.

In [11]:
def pretty_show_inference_example(question, reference_code, generated_code, *, title="Пример"):
    formatter = HtmlFormatter(
        noclasses=True,
        cssstyles="color: #111827; padding: 14px; border-radius: 10px; background: #f8fafc;",
        prestyles="color: inherit; background: transparent; margin: 0; line-height: 1.5;",
    )
    parts = [f"""
        <section style="box-sizing: border-box; max-width: 100%; padding: 20px;
                        margin: 16px 0; border: 1px solid #e2e8f0; border-radius: 16px;
                        background: #ffffff; color: #0f172a;">
            <h3 style="margin: 0 0 16px; color: #0f172a;">{escape(title)}</h3>
            <div style="box-sizing: border-box; max-width: 100%; padding: 18px 22px;
                        margin: 8px 0; border: 1px solid #cbd5e1; border-radius: 12px;
                        background: #f8fafc; color: #0f172a;">
                <div style="font-weight: 600; color: #64748b; margin-bottom: 10px;">
                    Вопрос
                </div>
                <div style="font-size: 16px; line-height: 1.6; white-space: normal;
                            overflow-wrap: anywhere;">{escape(question)}</div>
            </div>
    """]
    for label, code in (
        ("Эталонное решение:", reference_code),
        ("Наше решение:", generated_code),
    ):
        code_html = highlight(code, PythonLexer(), formatter)
        parts.append(f"""
            <h4 style="margin: 18px 0 8px; color: #0f172a;">{label}</h4>
            <div style="max-width: 100%; overflow-x: auto;">{code_html}</div>
        """)
    parts.append("</section>")
    display(HTML("".join(parts)))

## Модель и настройки

In [ ]:
DEVICE = "auto"

SAMPLE_SIZE = 5
SAMPLE_SEED = 62
MAX_TOKENS = 8192
MAX_NEW_TOKENS = 128

model, tokenizer = load_for_inference(EXPERIMENT_PATH, checkpoint=CHECKPOINT, device=DEVICE)
print(f"Модель: {type(model).__name__} | устройство: {next(model.parameters()).device}")

## Первый пример

In [13]:
manual_question = "Alice has 3 apples and buys 2 more. How many apples does she have?"
manual_reference = "def solve() -> int:\n    return 3 + 2\n"
manual_generated = generate_code(
    model,
    tokenizer,
    [manual_question],
    max_tokens=MAX_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
)[0]

pretty_show_inference_example(
    manual_question,
    manual_reference,
    manual_generated,
    title="Первый пример · яблоки",
)

## Примеры из валидации

Число примеров задаётся через `SAMPLE_SIZE`, выборка воспроизводится через `SAMPLE_SEED`.

In [14]:
val_data = load_from_disk(str(VAL_PATH))
val_indices = Random(SAMPLE_SEED).sample(
    range(len(val_data)), min(SAMPLE_SIZE, len(val_data))
)
val_sample = val_data.select(val_indices)

In [15]:
generated_codes = generate_code(
    model,
    tokenizer,
    val_sample,
    max_tokens=MAX_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
)

In [16]:
for number, (index, example, generated) in enumerate(
    zip(val_indices, val_sample, generated_codes), start=1
):
    pretty_show_inference_example(
        question=tokenizer.decode(example["input_ids"], skip_special_tokens=True),
        reference_code=tokenizer.decode(example["target_ids"], skip_special_tokens=True),
        generated_code=generated,
        title=f"Пример {number} · val[{index}]",
    )

Красота да и только. Выглядит прям вау!

## Рассчеты pass@1 на валидации

In [ ]:
from lib.evaluation import evaluate_pass_at_1

pass_at_1_result = evaluate_pass_at_1(
    model,
    tokenizer,
    val_data,
    max_tokens=MAX_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
    timeout=2.0,
    workers=4,
)

print(
    f"pass@1: {pass_at_1_result['pass_at_1']:.2%} "
    f"({pass_at_1_result['correct']} / {pass_at_1_result['total']})"
)
print(
    f"Пропущено строк без корректного эталона: {pass_at_1_result['skipped_invalid_answers']}; "
    f"повторов: {pass_at_1_result['skipped_duplicates']}"
)